In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('..')) 
from src.utils.config import USERNAME, ORGNAME
from src.utils.redivis_client import RedivisCatalog, display_hcup_variables # print_redivis_tree
from src.etl.ingest import fetch_hcup, fetch_hcup_ahal
from src.etl.transform import enrich_zip_data, enrich_fips_data

import redivis

# ==========================================
# WARNING: INTERACTIVE AUTHENTICATION
# ==========================================
# If no REDIVIS_API_TOKEN is set in your environment, Redivis will pause 
# execution here and output a URL in the terminal.
# You must click the URL and approve access via your web browser.
# 
# Upon success, Redivis securely caches your credentials locally in:
# Mac/Linux: ~/.redivis/python_credentials
# Windows:   C:\Users\YourUser\.redivis\python_credentials
#
# Future runs on this specific PC will use that hidden cache silently 
# without prompting you again, until the token expires or the folder is deleted.

# Trigger the authentication flow
print(f"Authenticating as `{USERNAME}`...\n")
if redivis.organization(ORGNAME).exists():  #  # <--- This forces to wait for authentication here
    print(f"\033[92m\u2714\033[0m Redivis Python Client `{USERNAME}` successfully authenticated")

Authenticating as `matteo_perini`...

✔ Redivis Python Client `matteo_perini` successfully authenticated


In [2]:
#hcup_cu=RedivisCatalog(ORGNAME)

#vars_df=display_hcup_variables(catalog = hcup_cu, state= 'New York', year= 2017, db_type= 'sedd')

#display(vars_df['Variable'])
#display(hcup_cu.datasets)

#display(hcup_cu.get_tables("kentucky_hcup:02jh:v3_1"))

#display(hcup_cu.get_variables("kentucky_hcup:02jh:v3_1","ky_sedd_2017_ahal:1hxt"))



In [4]:
#print_redivis_tree(ORGNAME, limit_tables=0) # default: limit_tables=3

In [3]:
#SEDD 

selected_cntys = {
    'Arizona': ('Pima, AZ', 'Pinal, AZ', 'Yavapai, AZ', 'Coconino, AZ'),  
    'Iowa': ('Polk, IA', 'Linn, IA', 'Scott, IA', 'Johnson, IA', 'Black Hawk, IA'), 
    'Kentucky': ('Jefferson, KY', 'Fayette, KY', 'Kenton, KY', 'Boone, KY', 'Warren, KY'), 
    'New Jersey': ('Bergen, NJ', 'Middlesex, NJ', 'Essex, NJ', 'Hudson, NJ', 'Monmouth, NJ'), 
    'New York': ('New York, NY', 'Kings, NY', 'Queens, NY', 'Bronx, NY', 'Richmond, NY'), 
    'North Carolina': ('Wake, NC', 'Mecklenburg, NC', 'Guilford, NC', 'Forsyth, NC', 'Cumberland, NC'), 
    'Utah': ('Salt Lake, UT', 'Utah, UT', 'Davis, UT', 'Weber, UT', 'Washington, UT')
}

hcup_cu=RedivisCatalog(ORGNAME)
db_type="sedd"





years_icd9=[str(y) for y in range(2006, 2015)] + ["2015q1q3"]
cols_to_pull_icd9=["KEY", "ZIP", "AMONTH", "AYEAR", "YEAR",  "DSHOSPID"]
prefix_icd9="ECODE"
vals_icd9=[f"E{i}" for i in range(950, 960)]

years_icd10=["2015q4"] + [str(y) for y in range(2016, 2021)]
cols_to_pull_icd10=["KEY", "ZIP", "AMONTH", "AYEAR", "YEAR",  "DSHOSPID"]
prefix_icd10="I10_DX"


# 1. Suicidal Ideation and Unspecified Attempts (Exact matches)
# R45851: Suicidal ideation
# T1491xx: Suicide attempt (Initial, Subsequent, Sequela)
vals_ideation_attempt = ['R45851', 'T1491XA', 'T1491XD', 'T1491XS']

# 2. Intentional Self-Harm by mechanism (Base codes X71 through X83)
vals_self_harm_x = [f"X{i}" for i in range(71, 84)]
# 3. Intentional Self-Poisoning (Base codes T36 through T50)
vals_poisoning_t = [f"T{i}" for i in range(36, 51)]

# Combine them all into one master tuple of prefixes
# (Tuples are required withpandas .str.startswith())
vals_icd10 = tuple(vals_ideation_attempt + vals_self_harm_x + vals_poisoning_t)


In [ ]:
#test 1 year 1 state
sedd_data = {}

# Ensure this matches the dictionary you defined earlier
selected_cntys = {
    'New York': ('New York, NY', 'Kings, NY', 'Queens, NY', 'Bronx, NY', 'Richmond, NY')
}

for state in selected_cntys.keys():
    print(f"--- Processing {state} ---")
    
    # 1. Fetch data
    df_icd9 = fetch_hcup(hcup_cu, state, db_type, ["2015q1q3"], cols_to_pull_icd9, prefix_icd9, vals_icd9, return_icd_cols=False)
    df_icd10 = fetch_hcup(hcup_cu, state, db_type, ["2015q4"], cols_to_pull_icd10, prefix_icd10, vals_icd10, return_icd_cols=False)
    df_ahal = fetch_hcup_ahal(hcup_cu, state, ["2015q1q3", "2015q4"])

    # 2. Robust DSHOSPID Cleaning
    for df in [df_icd9, df_icd10, df_ahal]:
        if 'DSHOSPID' in df.columns:
            # - Convert to string and remove '.0' float artifacts
            # - Strip whitespace
            # - Remove leading zeros (e.g., '0001' -> '1') EXCEPT if it's the only character ('0' stays '0')
            df['DSHOSPID'] = (df['DSHOSPID']            
            .astype(str)
            .str.replace(r'\.0$', '', regex=True)
            .str.strip()
            .str.replace(r'^0+(?!$)', '', regex=True))

    # 3. Robust Year Cleaning
    # Redivis can sometimes return AYEAR as 2015.0 and AHAL YEAR as 2015
    if 'AYEAR' in df_icd9.columns:
        df_icd9['AYEAR'] = df_icd9['AYEAR'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    if 'AYEAR' in df_icd10.columns:
        df_icd10['AYEAR'] = df_icd10['AYEAR'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    if 'YEAR' in df_ahal.columns:
        df_ahal['YEAR'] = df_ahal['YEAR'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

    # 4. Merge AHA Linkage Data
    # Using left_on and right_on to map the SEDD 'AYEAR' to the AHAL 'YEAR'
    df_icd9 = df_icd9.merge(df_ahal, left_on=['DSHOSPID', 'AYEAR'], right_on=['DSHOSPID', 'YEAR'], how='left')
    df_icd10 = df_icd10.merge(df_ahal, left_on=['DSHOSPID', 'AYEAR'], right_on=['DSHOSPID', 'YEAR'], how='left')

    # 5. Enrich with Geographic Data (FIPS)
    # Uses the Census CSV function established in transform.py
    df_icd9 = enrich_fips_data(df_icd9)
    df_icd10 = enrich_fips_data(df_icd10)

    # 6. Combine ICD-9 and ICD-10 and store in dictionary
    sedd_data[state] = pd.concat([df_icd9, df_icd10], ignore_index=True)

# Test the output for New York
ny_df = sedd_data['New York']
dshospid_table = ny_df['DSHOSPID'].value_counts()
print(dshospid_table)

#TODO: AGGREGATE AND DO SOME PRINTING TO BE SURE EVERYTHIGN IS RESONABLE AT COUNTY LEVEL



--- Processing New York ---
Fetching New York sedd 2015q1q3...


  0%|          | 0/9479 [00:00<?, ?it/s]

Fetching New York sedd 2015q4...


  0%|          | 0/20953 [00:00<?, ?it/s]

Fetching New York AHAL 2015...


  0%|          | 0/214 [00:00<?, ?it/s]

  0%|          | 0/222 [00:00<?, ?it/s]

  0%|          | 0/222 [00:00<?, ?it/s]

  0%|          | 0/214 [00:00<?, ?it/s]

DSHOSPID
413     4632
210     3371
630     3327
1178    2411
1       2383
        ... 
9691      14
482        8
9175       6
BLNK       2
4100       1
Name: count, Length: 200, dtype: int64


In [ ]:
sedd_data = {}
selected_cntys = {    'New York': ('New York, NY', 'Kings, NY', 'Queens, NY', 'Bronx, NY', 'Richmond, NY')}
for state in selected_cntys.keys():

    # Fetch and enrich ICD-9
    df_icd9 = fetch_hcup(hcup_cu, state, db_type, years_icd9, cols_to_pull_icd9, prefix_icd9, vals_icd9, return_icd_cols=False)
    #df_icd9 = enrich_zip_data(df_icd9)
    # Fetch and enrich ICD-10
    df_icd10 = fetch_hcup(hcup_cu, state, db_type, years_icd10, cols_to_pull_icd10, prefix_icd10, vals_icd10, return_icd_cols=False)
    #df_icd10 = enrich_zip_data(df_icd10)
    df_ahal = fetch_hcup_ahal(hcup_cu, state, years_icd9+years_icd10)

    # Merge inside the loop
    df_icd9 = df_icd9.merge(df_ahal, on=['DSHOSPID', 'YEAR'], how='left')
    df_icd10 = df_icd10.merge(df_ahal, on=['DSHOSPID', 'YEAR'], how='left')

    #add FIPS
    df_icd9 = enrich_fips_data(df_icd9)
    df_icd10 = enrich_fips_data(df_icd10)



    # Store one clean DataFrame per state
    sedd_data[state] = pd.concat([df_icd9, df_icd10], ignore_index=True)

In [ ]:
'''
state_name = "New Jersey"
db_type="sid"
sid_nj_icd9 = fetch_hcup(hcup_cu,state_name,db_type,years_icd9,cols_to_pull_icd9,prefix_icd9, vals_icd9, return_icd_cols=False)
sid_nj_icd9 = enrich_zip_data(sid_nj_icd9)
sid_nj_icd10 = fetch_hcup(hcup_cu,state_name,db_type,years_icd10,cols_to_pull_icd10,prefix_icd10, vals_icd10, return_icd_cols=False)
sid_nj_icd10 = enrich_zip_data(sid_nj_icd10)
'''